In [5]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
import datetime

# Import model builders
from Module.Model_isolated_sign import build_isolated_sign_model
from Module.Build_model_ctc import build_ctc_sign_model, CTCLossLayer


# Tạo thư mục logs
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Dataset loader helpers
def load_isolated_dataset(path):
    """Trả về (X, y) cho phase1. X: (samples, T, D), y: one-hot"""
    X, y = [], []
    labels = sorted(os.listdir(path))
    for i, label in enumerate(labels):
        label_dir = os.path.join(path, label)
        if not os.path.isdir(label_dir): continue
        for f in os.listdir(label_dir):
            if f.endswith(".npy"):
                arr = np.load(os.path.join(label_dir, f))
                X.append(arr)
                y.append(i)
    y = to_categorical(y, num_classes=len(labels))
    return X, y, labels


def pad_sequences_to_fixed_length(X, max_frames):
    """Pad hoặc truncate về cùng độ dài."""
    padded = []
    for arr in X:
        if arr.shape[0] > max_frames:
            arr = arr[:max_frames]
        elif arr.shape[0] < max_frames:
            pad = np.zeros((max_frames - arr.shape[0], arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        padded.append(arr)
    return np.array(padded, dtype=np.float32)


def load_ctc_dataset(path, max_frames=160):
    """Chuẩn bị dữ liệu cho phase2 (CTC)."""
    X, y_seq, input_len, label_len = [], [], [], []
    labels = sorted(os.listdir(path))
    for i, label in enumerate(labels):
        label_dir = os.path.join(path, label)
        if not os.path.isdir(label_dir): continue
        for f in os.listdir(label_dir):
            if f.endswith(".npy"):
                arr = np.load(os.path.join(label_dir, f))
                if arr.shape[0] > max_frames:
                    arr = arr[:max_frames]
                elif arr.shape[0] < max_frames:
                    pad = np.zeros((max_frames - arr.shape[0], arr.shape[1]), dtype=np.float32)
                    arr = np.concatenate([arr, pad], axis=0)
                X.append(arr)
                y_seq.append([i])
                input_len.append(max_frames)
                label_len.append(1)
    X = np.array(X, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.int32)
    input_len = np.expand_dims(np.array(input_len, dtype=np.int32), 1)
    label_len = np.expand_dims(np.array(label_len, dtype=np.int32), 1)
    return X, y_seq, input_len, label_len, labels

# Phase 1: Isolated Sign Recognition
def train_phase1(train_path="data/train", val_path="data/val",
                 model_dir="models/phase1", num_epochs=20, max_frames=180):

    print("\nBắt đầu Phase 1: Huấn luyện isolated sign model...")
    os.makedirs(model_dir, exist_ok=True)

    X_train, y_train, labels = load_isolated_dataset(train_path)
    X_val, y_val, _ = load_isolated_dataset(val_path)

    X_train = pad_sequences_to_fixed_length(X_train, max_frames)
    X_val = pad_sequences_to_fixed_length(X_val, max_frames)

    model = build_isolated_sign_model(num_keypoints=X_train.shape[2],
                                      num_classes=len(labels),
                                      max_frames=max_frames)
    model.summary()

    callbacks = [
        ModelCheckpoint(os.path.join(model_dir, "best_model.h5"),
                        save_best_only=True, monitor="val_accuracy", mode="max"),
        EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
        tensorboard_callback
    ]

    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=num_epochs,
                        batch_size=16,
                        callbacks=callbacks)

    # Lưu mô hình
    model.save(os.path.join(model_dir, "final_model.h5"))
    labels_path = os.path.join(model_dir, "labels.txt")
    with open(labels_path, "w", encoding="utf-8") as f:
        for lbl in labels:
            f.write(lbl + "\n")

    # Lưu history
    np.save(os.path.join(model_dir, "history.npy"), history.history)
    with open(os.path.join(model_dir, "history.json"), "w") as f:
        json.dump(history.history, f, indent=4)

    print(f"Phase 1 hoàn tất. Mô hình lưu tại: {model_dir}")
    return model, labels, history


# Phase 2: Continuous Sign (CTC)
def train_phase2(train_path="data/phase2_train",
                 model_dir="models/phase2",
                 pretrained_encoder_weights="models/phase1/final_model.h5",
                 num_epochs=25, max_frames=160):

    print("\nBắt đầu Phase 2: Huấn luyện continuous sign model (CTC)...")
    os.makedirs(model_dir, exist_ok=True)

    X_train, y_train, input_len, label_len, labels = load_ctc_dataset(train_path, max_frames)

    base_model = build_ctc_sign_model(num_keypoints=X_train.shape[2],
                                      vocab_size=len(labels),
                                      max_frames=max_frames)

    # Load encoder weight từ Phase 1 (nếu có)
    if os.path.exists(pretrained_encoder_weights):
        print(f"Đang load trọng số từ Phase 1: {pretrained_encoder_weights}")
        base_model.load_weights(pretrained_encoder_weights, by_name=True, skip_mismatch=True)
    else:
        print("Không tìm thấy pretrained weights, train lại từ đầu.")

    labels_in = tf.keras.Input(name="labels", shape=(None,), dtype="int32")
    input_len_in = tf.keras.Input(name="input_length", shape=(1,), dtype="int32")
    label_len_in = tf.keras.Input(name="label_length", shape=(1,), dtype="int32")

    y_pred = base_model.output
    loss_out = CTCLossLayer()([y_pred, labels_in, input_len_in, label_len_in])
    model = tf.keras.Model(inputs=[base_model.input, labels_in, input_len_in, label_len_in],
                           outputs=loss_out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))

    callbacks = [
        ModelCheckpoint(os.path.join(model_dir, "best_model.h5"),
                        save_best_only=True, monitor="loss", mode="min"),
        EarlyStopping(monitor="loss", patience=5, restore_best_weights=True),
        tensorboard_callback
    ]

    history = model.fit(x=[X_train, y_train, input_len, label_len],
                        y=np.zeros(len(X_train)),
                        batch_size=8,
                        epochs=num_epochs,
                        callbacks=callbacks)

    # Lưu mô hình
    base_model.save(os.path.join(model_dir, "final_model_encoder.h5"))
    labels_path = os.path.join(model_dir, "labels.txt")
    with open(labels_path, "w", encoding="utf-8") as f:
        for lbl in labels:
            f.write(lbl + "\n")

    # Lưu history
    np.save(os.path.join(model_dir, "history.npy"), history.history)
    with open(os.path.join(model_dir, "history.json"), "w") as f:
        json.dump(history.history, f, indent=4)

    print(f"Phase 2 hoàn tất. Mô hình lưu tại: {model_dir}")
    return base_model, labels, history


# MAIN
if __name__ == "__main__":
    mode = input("Chọn phase để huấn luyện ( isolated(1) / continuous(2) ): ").strip().lower()

    if mode == "1":
        train_phase1(train_path="Data_Keypoints/train", val_path="Data_Keypoints/val",
                     model_dir="models/phase1", num_epochs=30, max_frames=180)

    elif mode == "2":
        train_phase2(train_path="data/phase2_train",
                     model_dir="models/phase2",
                     pretrained_encoder_weights="models/phase1/final_model.h5",
                     num_epochs=25, max_frames=180)
    else:
        print("Vui lòng nhập 'isolated' hoặc 'continuous'.")



Bắt đầu Phase 1: Huấn luyện isolated sign model...
Model: "IsolatedSignModel"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 keypoints_input (InputLayer)   [(None, 180, 128)]   0           []                               
                                                                                                  
 dense_28 (Dense)               (None, 180, 256)     33024       ['keypoints_input[0][0]']        
                                                                                                  
 tf.__operators__.add_20 (TFOpL  (None, 180, 256)    0           ['dense_28[0][0]']               
 ambda)                                                                                           
                                                                                                  
 multi_head_attention_8 (Multi